# EZhire Gradio Setup
Loads saved models and metrics artifacts to run the dashboard.

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
import nltk
import fitz
import torch
import joblib
from collections import Counter
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# wordninja is optional: if it's unavailable, desegment() still does the cheap
# camelCase / letter-digit repairs and just skips the blob-splitting fallback.
try:
    import wordninja
    _HAS_WORDNINJA = True
except Exception:
    _HAS_WORDNINJA = False

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
ARTIFACT_DIR = os.path.join(BASE_DIR, "artifacts")

STOP_WORDS = set(stopwords.words("english"))
print(f"Device: {DEVICE}")

## Load config and all three SBERT models

In [ ]:
CONFIG_PATH = os.path.join(ARTIFACT_DIR, 'ensemble_config.json')
best_sbert_name = 'bert'   # Model C: best on Pearson/MAE/RMSE/R2 (overridden by config if present)

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        cfg = json.load(f)
    best_sbert_name = cfg.get('best_sbert_name', best_sbert_name)
else:
    print('Warning: ensemble_config.json not found, using defaults.')

SBERT_PATHS = {
    'mpnet': os.path.join(MODEL_DIR, 'ezhire-mpnet_files'),
    'roberta': os.path.join(MODEL_DIR, 'ezhire-roberta-base_files'),
    'bert': os.path.join(MODEL_DIR, 'ezhire-bert-base_files')
}

# Load all three SBERT models so we can switch between them at inference time.
sbert_models = {}
for name, path in SBERT_PATHS.items():
    if os.path.exists(os.path.join(path, 'modules.json')):
        m = SentenceTransformer(path, device=DEVICE)
        m.max_seq_length = 384
        sbert_models[name] = m
        print(f'Loaded SBERT: {name}')
    else:
        print(f'Skipping {name}: model not found at {path}')

if not sbert_models:
    raise FileNotFoundError('No SBERT models found. Run the training notebooks first.')

MODEL_NAMES = list(sbert_models.keys())
if best_sbert_name not in MODEL_NAMES:
    best_sbert_name = MODEL_NAMES[0]
print(f'Available models: {MODEL_NAMES} | default: {best_sbert_name}')

## Scoring helpers

In [ ]:
import zipfile

CHUNK_OVERLAP = 38
MAX_RESUME_CHUNKS = 10
MAX_JD_CHUNKS = 8
TOP_K_CHUNK_PAIRS = 5
ENCODE_BATCH = 16 if DEVICE == 'cuda' else 8

# Corpus-fit TF-IDF vectorizer saved by the metrics notebook. If present we reuse
# its trained IDF (meaningful across the whole corpus); otherwise we fall back to
# the old per-document behaviour so the app still runs standalone.
TFIDF_UNION_PATH = os.path.join(ARTIFACT_DIR, 'tfidf_union.joblib')
try:
    TFIDF_UNION = joblib.load(TFIDF_UNION_PATH)
    _word_vec = dict(TFIDF_UNION.transformer_list).get('word')
    KW_IDF = (dict(zip(_word_vec.get_feature_names_out(), _word_vec.idf_))
              if _word_vec is not None else None)
    print(f'Loaded corpus-fit TF-IDF vectorizer ({len(KW_IDF) if KW_IDF else 0} word terms)')
except Exception as e:
    TFIDF_UNION, KW_IDF = None, None
    print(f'No corpus-fit TF-IDF vectorizer ({e}); using per-document fallback.')

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

# --- Word desegmentation (mirrors the metrics notebook) ---------------------
# Repairs PDF-extraction fusions ("SummaryI", "VistaWindows", "98Windows") so the
# lexical keyword matching sees real words. Cheap regex splits + an optional
# wordninja pass for long all-lowercase blobs (acronyms/short tokens untouched).
_CAMEL = re.compile(r'(?<=[a-z])(?=[A-Z])')
_LET_DIG = re.compile(r'(?<=[A-Za-z])(?=\d)|(?<=\d)(?=[A-Za-z])')

def desegment(text):
    if not isinstance(text, str): text = str(text)
    text = _CAMEL.sub(' ', text)
    text = _LET_DIG.sub(' ', text)
    out = []
    for tok in text.split():
        if _HAS_WORDNINJA and tok.isalpha() and tok.islower() and len(tok) > 14:
            out.extend(wordninja.split(tok))
        else:
            out.append(tok)
    return ' '.join(out)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', desegment(text).lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t) > 1)

def light_clean(text):
    """Desegmented + lowercased text with punctuation kept - matches the
    preprocessing the saved TF-IDF vectorizer was fitted on."""
    t = desegment(raw_text(text)).lower()
    return re.sub(r'\s+', ' ', t).strip()

def extract_first_sentence(text, max_chars=150):
    t = raw_text(text)
    m = re.search(r'(?<=[a-zA-Z0-9])[.!?]', t)
    if m and m.start() > 10:
        sentence = t[:m.start() + 1].strip()
    else:
        sentence = t[:max_chars].strip()
    return sentence[:max_chars]

def token_chunk_body(body, tokenizer, max_tokens=512, overlap=CHUNK_OVERLAP, prefix_tokens=0):
    body = raw_text(body)
    if not body:
        return []
    ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)
    effective_max = max(64, max_tokens - prefix_tokens - 2)
    if len(ids) <= effective_max:
        return [body]
    overlap = min(overlap, effective_max // 2)
    step = effective_max - overlap
    chunks = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + effective_max]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + effective_max >= len(ids):
            break
    return chunks

def make_text_chunks(text, tokenizer, model_max_tokens=512, overlap=CHUNK_OVERLAP,
                     max_chunks=MAX_RESUME_CHUNKS, source_label='DOCUMENT'):
    text = raw_text(text)
    if not text:
        return []
    doc_ctx = extract_first_sentence(text)
    prefix = f'[DOC]: {doc_ctx} [{source_label}] ' if doc_ctx else f'[{source_label}] '
    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))
    body_chunks = token_chunk_body(
        text, tokenizer,
        max_tokens=model_max_tokens,
        overlap=overlap,
        prefix_tokens=prefix_tokens
    )
    chunks = [f'{prefix}{c}' for c in body_chunks]
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]
    fallback = text[:2000]
    return chunks or [f'{prefix}{fallback}']

def aggregate_chunk_sims(sims, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sims, float)
    if sims.size == 0: return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def score_doc_pair(model, t1, t2, left='RESUME', right='JOB'):
    tok = model.tokenizer
    mlen = model.max_seq_length
    c1 = make_text_chunks(t1, tok, model_max_tokens=mlen,
                          max_chunks=MAX_RESUME_CHUNKS, source_label=left)
    c2 = make_text_chunks(t2, tok, model_max_tokens=mlen,
                          max_chunks=MAX_JD_CHUNKS, source_label=right)
    e1 = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    e2 = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_sims(sims)

def tfidf_score(t1, t2):
    """Cosine similarity of two documents. Uses the corpus-fit vectorizer (real IDF)
    when available; otherwise falls back to a per-pair fit."""
    try:
        if TFIDF_UNION is not None:
            m = TFIDF_UNION.transform([t1, t2])
        else:
            m = TfidfVectorizer().fit_transform([t1, t2])
        return float(cosine_similarity(m[0:1], m[1:2])[0][0])
    except Exception:
        return 0.0

def get_tier(score):
    if score >= 70: return 'Strong Match'
    if score >= 45: return 'Potential Fit'
    return 'Poor Match'

def extract_pdf_text(path):
    try:
        with fitz.open(path) as doc:
            return ' '.join(p.get_text() for p in doc).strip()
    except Exception as e:
        return f'PDF error: {e}'

def extract_pdf_text_from_bytes(data):
    try:
        with fitz.open(stream=data, filetype='pdf') as doc:
            return ' '.join(p.get_text() for p in doc).strip()
    except Exception as e:
        return f'PDF error: {e}'

def extract_resumes_from_files(files):
    """Accept uploaded PDFs and/or ZIP archives of PDFs; return [(name, text), ...]."""
    resumes = []
    for f in (files or []):
        path = f.name
        low = path.lower()
        if low.endswith('.zip'):
            try:
                with zipfile.ZipFile(path) as zf:
                    for info in zf.namelist():
                        if info.endswith('/') or not info.lower().endswith('.pdf'):
                            continue
                        base = os.path.basename(info)
                        if not base or base.startswith('.') or info.startswith('__MACOSX'):
                            continue
                        text = extract_pdf_text_from_bytes(zf.read(info))
                        resumes.append((os.path.splitext(base)[0], text))
            except Exception as e:
                resumes.append((os.path.basename(path), f'ZIP error: {e}'))
        elif low.endswith('.pdf'):
            text = extract_pdf_text(path)
            resumes.append((os.path.splitext(os.path.basename(path))[0], text))
    return resumes

def extract_keywords(text, n=30):
    """Top-n keywords. Ranks by tf * corpus-IDF (from the saved vectorizer) over
    desegmented text when available; otherwise falls back to a per-document fit."""
    text = desegment(text)
    try:
        if KW_IDF is not None:
            toks = [t for t in word_tokenize(re.sub(r'[^a-z0-9\s]', ' ', text.lower()))
                    if t not in STOP_WORDS and len(t) > 2]
            tf = Counter(toks)
            scored = {w: c * KW_IDF[w] for w, c in tf.items() if w in KW_IDF}
            top = sorted(scored, key=scored.get, reverse=True)[:n]
            if top:
                return set(top)
        vec = TfidfVectorizer(stop_words='english', max_features=n)
        vec.fit([text])
        return set(vec.get_feature_names_out())
    except Exception:
        tokens = word_tokenize(text.lower())
        return set(w for w, _ in Counter(
            t for t in tokens if t not in STOP_WORDS and len(t) > 2
        ).most_common(n))

def keyword_html(resume, jd):
    kr = extract_keywords(resume, 40)
    kj = extract_keywords(jd, 40)
    matched, missing, extra = kr & kj, kj - kr, kr - kj
    return (
        "<div style='font-family:sans-serif;padding:12px'>"
        "<h3 style='color:#27AE60'>Matched (" + str(len(matched)) + ")</h3>"
        "<p style='color:#27AE60'>" + (', '.join(sorted(matched)) or 'None') + "</p>"
        "<hr><h3 style='color:#E74C3C'>Missing from Resume (" + str(len(missing)) + ")</h3>"
        "<p style='color:#E74C3C'>" + (', '.join(sorted(missing)) or 'None') + "</p>"
        "<hr><h3 style='color:#F39C12'>Resume-Only (" + str(len(extra)) + ")</h3>"
        "<p style='color:#F39C12'>" + (', '.join(sorted(extra)) or 'None') + "</p></div>"
    )

def sbert_score(model_name, resume, jd):
    return score_doc_pair(sbert_models[model_name], resume, jd)

def score_single(model_name, resume, jd):
    """Return (sbert_score_pct, tfidf_score_pct) for a single model. No ensemble.
    The displayed score is the SBERT score; tfidf is informational."""
    s = sbert_score(model_name, resume, jd)
    t = tfidf_score(light_clean(resume), light_clean(jd))
    return round(s * 100, 2), round(t * 100, 2)

## Load metrics artifacts (optional)

In [4]:
metrics_path = os.path.join(ARTIFACT_DIR, 'metrics_df.csv')
scores_path = os.path.join(ARTIFACT_DIR, 'validation_scores.csv')

if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()
    print('Warning: metrics_df.csv not found, comparison plots will be empty.')

if os.path.exists(scores_path):
    df_sample = pd.read_csv(scores_path)
else:
    df_sample = pd.DataFrame()
    print('Warning: validation_scores.csv not found, heatmaps will be empty.')

In [ ]:
def score_resumes(model_name, files, paste, jd):
    """Score one or more resumes (PDFs and/or ZIPs of PDFs) with the chosen model."""
    if not jd or not jd.strip():
        return 'Please enter a job description.', None, None, ''
    resumes = extract_resumes_from_files(files)
    if paste and paste.strip():
        resumes.append(('Pasted Resume', paste.strip()))
    if not resumes:
        return 'Please upload resume PDFs/ZIPs or paste resume text.', None, None, ''

    rows = []
    for name, text in resumes:
        if not text or text.startswith(('PDF error', 'ZIP error')):
            rows.append(dict(Candidate=name, Score=0.0, Tier='Read error'))
            continue
        s, _ = score_single(model_name, text, jd)
        rows.append(dict(Candidate=name, Score=s, Tier=get_tier(s)))

    rdf = pd.DataFrame(rows).sort_values('Score', ascending=False).reset_index(drop=True)
    rdf.insert(0, 'Rank', range(1, len(rdf) + 1))

    colors = ['#27AE60' if r >= 70 else '#F39C12' if r >= 45 else '#E74C3C'
              for r in rdf['Score']]
    fig = go.Figure(go.Bar(x=rdf['Candidate'], y=rdf['Score'],
                           marker_color=colors,
                           text=[f'{v:.1f}%' for v in rdf['Score']],
                           textposition='outside'))
    fig.update_layout(title=f'{model_name} - Candidate Scores',
                      yaxis=dict(range=[0, 115], title='Match %'),
                      height=400, plot_bgcolor='rgba(0,0,0,0)',
                      paper_bgcolor='rgba(0,0,0,0)')

    summary = f'### Model: `{model_name}`  |  {len(rdf)} resume(s) scored'
    # Keyword overlap is only meaningful for a single resume.
    kw = ''
    if len(resumes) == 1 and not resumes[0][1].startswith(('PDF error', 'ZIP error')):
        kw = keyword_html(resumes[0][1], jd)

    return summary, rdf[['Rank', 'Candidate', 'Score', 'Tier']], fig, kw

def tab3_heatmap(n=20):
    if df_sample.empty:
        blank = go.Figure().update_layout(title='No validation scores loaded.')
        return blank, blank
    n = min(int(n), len(df_sample))
    sub = df_sample.head(n).copy()
    sub['Label'] = [f'C{i + 1}' for i in range(n)]
    cols = ['mpnet_score', 'roberta_score', 'bert_score', 'tfidf_score']
    z = sub[cols].values.T
    fh = go.Figure(go.Heatmap(z=z, x=sub['Label'].tolist(),
                                y=['mpnet', 'roberta', 'bert', 'TF-IDF'],
                                colorscale='RdYlGn', zmin=0, zmax=100,
                                text=np.round(z, 1), texttemplate='%{text}',
                                colorbar=dict(title='%')))
    fh.update_layout(title=f'Score Heatmap - {n} Candidates', height=380,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    fd = go.Figure()
    for col, nm, c in zip(cols,
                         ['mpnet', 'roberta', 'bert', 'TF-IDF'],
                         ['#4A90D9', '#9B59B6', '#E67E22', '#95A5A6']):
        fd.add_trace(go.Histogram(x=df_sample[col], name=nm, opacity=0.6,
                                   marker_color=c, nbinsx=20))
    fd.update_layout(barmode='overlay', title='Score Distribution', height=340,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    return fh, fd

def tab4_compare():
    if metrics_df.empty:
        blank = go.Figure().update_layout(title='No metrics loaded.')
        return blank, blank, blank
    df = metrics_df.copy()
    if 'model' in df.columns:
        df = df.set_index('model')
    # Compare the three models only - drop any ensemble row.
    models = [m for m in df.index.tolist() if str(m).lower() != 'ensemble']
    palette = ['#4A90D9', '#9B59B6', '#E67E22', '#95A5A6', '#27AE60']
    mcols = ['spearman', 'pearson', 'ndcg', 'precision_at_5', 'precision_at_10', 'mrr']
    fb = go.Figure()
    for i, m in enumerate(models):
        vals = [df.loc[m, c] for c in mcols]
        fb.add_trace(go.Bar(name=m,
            x=[c.replace('_', ' ').title() for c in mcols],
            y=vals, marker_color=palette[i % len(palette)],
            text=[f'{v:.3f}' for v in vals], textposition='outside'))
    fb.update_layout(barmode='group', title='All Metrics Comparison',
                     yaxis=dict(range=[-0.2, 1.3]), height=430,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    rcols = ['spearman', 'ndcg', 'precision_at_5', 'precision_at_10', 'mrr', 'pearson']
    fr = go.Figure()
    for i, m in enumerate(models):
        vals = [df.loc[m, c] for c in rcols] + [df.loc[m, rcols[0]]]
        cats = [c.replace('_', ' ').title() for c in rcols + [rcols[0]]]
        fr.add_trace(go.Scatterpolar(r=vals, theta=cats, fill='toself',
                                      name=m, line_color=palette[i % len(palette)], opacity=0.5))
    fr.update_layout(polar=dict(radialaxis=dict(range=[0, 1])),
                     title='Radar Chart', height=430,
                     paper_bgcolor='rgba(0,0,0,0)')
    fm = go.Figure(go.Bar(x=models,
        y=[df.loc[m, 'mae'] for m in models],
        marker_color=palette[:len(models)],
        text=[f"{df.loc[m, 'mae']:.4f}" for m in models],
        textposition='outside'))
    fm.update_layout(title='MAE norm (lower=better)', height=350,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    return fb, fr, fm

print('Dashboard functions ready')

In [ ]:
fig_bar_cmp, fig_radar_cmp, fig_mae_cmp = tab4_compare()
fig_heatmap, fig_dist = tab3_heatmap(20)

# Most decision-relevant metrics for a ranking task (keep the table compact).
KEY_METRICS = ['model', 'spearman', 'pearson', 'ndcg', 'precision_at_5', 'mrr']

with gr.Blocks(theme=gr.themes.Soft(), title='EZhire') as demo:
    gr.Markdown('# EZhire - Resume-Job Semantic Similarity Scoring')

    with gr.Tab('Score Resumes'):
        # Job description first, then the resume upload.
        jd_box = gr.Textbox(label='Job Description', lines=7,
                            placeholder='Paste the job description here...')
        model_sel = gr.Radio(MODEL_NAMES, value=best_sbert_name,
                             label='Model')
        upload = gr.File(label='Upload resumes - PDF files or ZIP archives',
                         file_count='multiple', file_types=['.pdf', '.zip'])
        paste = gr.Textbox(label='Paste resume text', lines=6)
        btn = gr.Button('Score Resumes', variant='primary')

        out_md = gr.Markdown()
        with gr.Row():
            out_tbl = gr.Dataframe(label='Results', interactive=False)
            out_fig = gr.Plot(label='Scores')
        out_kw = gr.HTML(label='Keyword Overlap (single resume only)')

        btn.click(score_resumes, [model_sel, upload, paste, jd_box],
                  [out_md, out_tbl, out_fig, out_kw])

    with gr.Tab('Score Heatmap'):
        n_sl = gr.Slider(5, min(50, len(df_sample)) if not df_sample.empty else 50,
                         value=20, step=5, label='Candidates to show')
        btn3 = gr.Button('Refresh')
        h_plot = gr.Plot(value=fig_heatmap)
        d_plot = gr.Plot(value=fig_dist)
        btn3.click(tab3_heatmap, [n_sl], [h_plot, d_plot])

    with gr.Tab('Model Comparison'):
        gr.Markdown(
            'All three models evaluated on the full validation set against ATS ground truth.'
        )
        if not metrics_df.empty:
            _show = [c for c in KEY_METRICS if c in metrics_df.columns]
            gr.Dataframe(value=metrics_df[_show].round(3), label='Metrics Table')
        gr.Plot(value=fig_bar_cmp, label='All Metrics')
        gr.Plot(value=fig_radar_cmp, label='Radar Chart')
        gr.Plot(value=fig_mae_cmp, label='MAE')
        gr.Markdown(
            '**Spearman/Pearson**: correlation with ATS ground truth. '
            '**NDCG**: ranking quality. **Precision@5**: good fits in top-5. '
            '**MRR**: rank of the first strong match.'
        )

demo.launch(share=True, debug=True)